In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 1. Load the data
df = pd.read_csv('QB_Combine_Data.csv')

# 2. Parsing Function
def convert_to_inches(val):
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    try:
        v = str(val).strip()
        if "'" in v:
            parts = v.replace('"', '').split("'")
            return int(parts[0]) * 12 + (int(parts[1]) if parts[1] else 0)
        return float(v)
    except:
        return np.nan

# Clean core physicals
df['Height_In'] = df['Height'].apply(convert_to_inches)
df['Broad_In'] = df['Broad'].apply(convert_to_inches)

# Identify the PFF column
pff_col = [col for col in df.columns if 'PFF' in col.upper()][0]
df[pff_col] = pd.to_numeric(df[pff_col], errors='coerce')

# 3. Explicit Feature List (Excluding Wingspan, Bench, and 10 Split)
features = [
    'Height_In', 'Weight', 'Hands', 'Arms', 
    '40 Yd', 'Vertical', 'Broad_In', '3-Cone', 'Shuttle'
]

# Ensure all selected features are numeric
for feat in features:
    if feat in df.columns:
        df[feat] = pd.to_numeric(df[feat], errors='coerce')

# Drop rows missing the target variable (PFF)
df_model = df.dropna(subset=[pff_col]).copy()

# 4. Define x and y
y = df_model[pff_col]
x = df_model[features]

# 5. Preprocessing
# Fill missing drill results with the average (Imputation)
imputer = SimpleImputer(strategy='mean')
x_imputed = imputer.fit_transform(x)

# Normalize the data (Standardization) 
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_imputed)

# Rebuild the DataFrame for Statsmodels
x_final = pd.DataFrame(x_scaled, columns=features, index=df_model.index)
x_final = sm.add_constant(x_final)  # Add intercept

# 6. Run OLS Regression
model = sm.OLS(y, x_final).fit()
print(f"Analysis complete for {len(df_model)} QBs.")
print(model.summary())

"""
Comments and some code written by Google Gemini Pro 3.1
"""

Analysis complete for 26 QBs.
                              OLS Regression Results                              
Dep. Variable:      PFF_for_first_4_years   R-squared:                       0.442
Model:                                OLS   Adj. R-squared:                  0.128
Method:                     Least Squares   F-statistic:                     1.409
Date:                    Sat, 18 Apr 2026   Prob (F-statistic):              0.263
Time:                            18:46:57   Log-Likelihood:                -92.227
No. Observations:                      26   AIC:                             204.5
Df Residuals:                          16   BIC:                             217.0
Df Model:                               9                                         
Covariance Type:                nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------

'\nComments and some code written by Google Gemini Pro 3.1\n'